# Spreading Activation over the Small World of Words (SWOW) dataset
Loads the Small World of Words associative-strength data and runs a
spreading activation random walk for `N_STEPS` steps from `CUE_WORD`.
In the works: reinforcement learning algorithm that determines whether to stop or continue retrieval

## Setup

In [1]:
import random
from collections import defaultdict
from swow_activation import load_graph, activation_step
from swow_agent import Agent



## Configuration
Edit the variables below before running.

In [2]:
# Path to the SWOW associative-strength TSV
DATA_FILE    = "data/strengths_small.csv"

# Strength column: "R123.Strength" (all responses) or "R1.Strength" (first response only)
STRENGTH_COL = "R123.Strength"

# Starting cue word
CUE_WORD     = "apple"

# Number of steps to take
N_STEPS      = 100

# Random seed — set to an integer for reproducibility, or None for a random walk
SEED         = None
rng = random.Random(SEED)

## Load graph

In [3]:
graph = load_graph(DATA_FILE, strength_col=STRENGTH_COL)

n_nodes = len(graph)
n_edges = sum(len(v) for v in graph.values())
print(f"Graph loaded: {n_nodes:,} cue nodes, {n_edges:,} total edges.")

Graph loaded: 9,997 cue nodes, 865,260 total edges.


## Run walk

In [15]:
cue = CUE_WORD.strip().lower()

if cue not in graph:
    close = [k for k in graph if k.startswith(cue[:3])][:5]
    raise ValueError(f"'{cue}' not found in graph. Close matches: {close}")

path: list[str] = [cue]
dead_end_responses: dict[str, set[str]] = defaultdict(set)

step = 0
agent = Agent(cue, temp=0.2)
action = agent.policy(cue)

while step < N_STEPS and action:
    current = path[-1]

    next_node, exhausted = activation_step(graph, current, dead_end_responses, rng)

    if next_node is not None:
        # Successful step forward
        path.append(next_node)
        agent.v_learning(current, 1, next_node)
        step += 1
        action = agent.policy(next_node)

    elif exhausted:
        # Current node has no viable neighbours — backtrack
        print(f"  '{current}' is a dead end — backtracking...")
        if len(path) == 1:
            print(f"  Could not recover from dead end at starting cue '{current}'. Walk ends early.")
            break
        parent = path[-2]
        dead_end_responses[parent].add(current)
        path.pop()

print(f"\nWalk complete: {len(path) - 1} steps taken.")


Walk complete: 83 steps taken.


## Results

In [ ]:
print("=" * 55)
print(f"  Spreading activation walk  ({len(path) - 1} steps)")
print("=" * 55)
for i, node in enumerate(path):
    if i == 0:
        print(f"  [START]  {node}  (v={agent.vtable[node]})")
    else:
        prev = path[i - 1]
        strength = graph.get(prev, {}).get(node)
        strength_str = f"  (p={strength:.4f})" if strength is not None else ""
        print(f"  step {i:>3}  {node}{strength_str}  (v={agent.vtable[node]})")
print("=" * 55)
print(agent.vtable)

  Spreading activation walk  (83 steps)
  [START]  apple  (v=1.09)
  step   1  pie  (p=0.0574)  (v=1.09)
  step   2  yum  (p=0.0035)  (v=1.09)
  step   3  food  (p=0.1312)  (v=1.09)
  step   4  clothing  (p=0.0034)  (v=1.09)
  step   5  worn  (p=0.0102)  (v=1.09)
  step   6  tired  (p=0.0629)  (v=1.171)
  step   7  rest  (p=0.0138)  (v=1.0981)
  step   8  tired  (p=0.0269)  (v=1.171)
  step   9  old  (p=0.0069)  (v=1.171)
  step  10  retired  (p=0.0034)  (v=1.0981)
  step  11  old  (p=0.1916)  (v=1.171)
  step  12  wrinkle  (p=0.0034)  (v=1.09)
  step  13  crinkle  (p=0.0107)  (v=1.09)
  step  14  plastic  (p=0.0083)  (v=1.09)
  step  15  fake  (p=0.0352)  (v=1.09)
  step  16  identity  (p=0.0036)  (v=1.09)
  step  17  self  (p=0.0877)  (v=1.09)
  step  18  pen  (p=0.0035)  (v=1.09)
  step  19  red  (p=0.0135)  (v=1.09)
  step  20  bright  (p=0.0270)  (v=1.09)
  step  21  happy  (p=0.0102)  (v=1.171)
  step  22  satisfied  (p=0.0170)  (v=1.09)
  step  23  done  (p=0.0141)  (v=1.09)
  s